For selected electrodes, run stepwise regression to understand whether progressively refined features improve predictive power.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from copy import deepcopy
import itertools
from pathlib import Path
import pickle
import re

import matplotlib.pyplot as plt
import mne
from mne.decoding import ReceptiveField
import numpy as np
import pandas as pd
from sklearn.base import clone, BaseEstimator, TransformerMixin, check_is_fitted
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import mean_squared_error
import seaborn as sns
from tqdm.auto import tqdm
from scipy import stats

from src.data import add_metadata_features
from src.stimuli import POD_dict
from src.models.trf import estimate_trf, TRF

In [ ]:
epochs_path = "outputs/epochs_preprocessed/EC248_epo.fif"
trf_path = "outputs/trfs/EC248/results.pkl"
eois_path = "outputs/trf_eois/eois.csv"
outdir = "."

feature_progressions = {
    "acoustic": [
        ("categorical_acoustic_cue", "onset"),
        ("linear_acoustic_cue", "onset"),
    ],

    "mismatch": [
        # ("mismatch", "PoD"),
        ("mismatch_left_right", "PoD"),
        ("belief_update", "PoD"),
    ],
}

num_folds = 2
Cs = np.logspace(-4, 2, 7)

In [ ]:
subject = Path(trf_path).parent.name

In [ ]:
eoi_df = pd.read_csv(eois_path)
# only retain current subject electrodes
eoi_df = eoi_df[eoi_df.subject == subject]
# only retain feature blocks showing positive UV
eoi_df = eoi_df[eoi_df.unique_variance > 0]
# only retain feature blocks which we are interested in for stepwise regression
eoi_df = eoi_df[eoi_df.feature_block.isin(feature_progressions.keys())]
eoi_df

In [ ]:
if eoi_df.empty:
    print("No target EOIs found for this subject. All good. Stop.")
    import sys; sys.exit(0)

In [ ]:
mne.set_config("MNE_TQDM", "off")

In [ ]:
ep = mne.read_epochs(epochs_path)

In [ ]:
old_metadata = ep.metadata.copy()
ep.metadata = add_metadata_features(old_metadata)

In [ ]:
with open(trf_path, "rb") as f:
    trf_data = pickle.load(f)

In [ ]:
features_per_phoneme_pair = trf_data["features_per_phoneme_pair"]
feature_blocks = trf_data["feature_blocks"]

In [ ]:
assert set(feature_progressions.keys()) <= set(feature_blocks.keys())

all_feature_progression_features = set(feature for features in feature_progressions.values() for feature, _ in features)
all_trf_features = set(feat for feat, _ in features_per_phoneme_pair)
new_features = all_feature_progression_features - all_trf_features

print("NB, will be adding new features beyond existing TRF fits:")
new_features

In [ ]:
def fit_stepwise(feature_block, fit_electrodes=None):
    md = ep.metadata
    epoch_idxs = md.index.values

    estimator = ReceptiveField(tmin=-0.1, tmax=0.7, sfreq=ep.info["sfreq"])
    outer_cv = StratifiedKFold(num_folds, shuffle=True, random_state=42)
    inner_cv = StratifiedKFold(num_folds, shuffle=True, random_state=42)

    # # DEV
    # epoch_idxs = epoch_idxs[:200]
    # estimator = ReceptiveField(tmin=-0.1, tmax=0.1, sfreq=ep.info["sfreq"])

    param_grid = {"estimator__estimator": Cs}

    from sklearn.model_selection import GridSearchCV, cross_validate
    from sklearn.metrics import r2_score, make_scorer
    stratify_class = md.loc[epoch_idxs].stratify_class

    cv_results = []
    for i in range(len(feature_progressions[feature_block]) + 1):
        fit_block_features = feature_progressions[feature_block][:i]
        skip_block_features = feature_progressions[feature_block][i:]
        
        fit_features = list(set(features_per_phoneme_pair + fit_block_features) - set(skip_block_features))
        print(fit_features)
        
        pipeline_i = TRF(estimator, ep, fit_features, fit_electrodes=fit_electrodes)
        print("\t", pipeline_i.feature_names)
        clf_i = GridSearchCV(pipeline_i, param_grid, cv=inner_cv, n_jobs=1)
        
        cv_results_i = cross_validate(clf_i, X=epoch_idxs, y=stratify_class,
                                      cv=outer_cv, n_jobs=2,
                                      return_estimator=True,
                                      return_train_score=True,
                                      return_indices=True,
                                      verbose=100)

        Y_pred, Y_true = [], []
        for (est_ij, test_indices_ij) in zip(cv_results_i["estimator"], cv_results_i["indices"]["test"]):
            _, Y_ij = est_ij.best_estimator_._prepare_design_matrix(test_indices_ij)
            Y_pred_ij = est_ij.best_estimator_.predict(test_indices_ij)
            Y_pred.append(Y_pred_ij)
            Y_true.append(Y_ij)

        Y_pred = np.concatenate(Y_pred)
        Y_true = np.concatenate(Y_true)

        # number of features per output channel
        num_total_features = np.prod(est_ij.best_estimator_.coef_.shape[1:])

        cv_results.append({
            "feature_progression": feature_block,
            "progression_step": i,
            "fit_block_features": fit_block_features,
            "skip_block_features": skip_block_features,

            "cv_results": cv_results_i,

            "Y_pred": Y_pred,
            "Y_true": Y_true,
            "num_total_features": num_total_features,
        })

    return cv_results


def ftest_stepwise(feature_block, cv_results):
    ftest_results = []

    for step1, step2 in zip(cv_results, cv_results[1:]):
        # print(step1["fit_block_features"], step2["fit_block_features"])
        # print(step1["num_total_features"], step2["num_total_features"])
        Y_pred1, Y_true1 = step1["Y_pred"], step1["Y_true"]
        Y_pred2, Y_true2 = step2["Y_pred"], step2["Y_true"]
        assert Y_pred1.shape == Y_pred2.shape
        assert Y_true1.shape == Y_true2.shape

        rss_1 = mean_squared_error(Y_true1, Y_pred1, multioutput="raw_values")
        rss_2 = mean_squared_error(Y_true2, Y_pred2, multioutput="raw_values")

        # calculate dof
        n = Y_true1.shape[0]
        p1 = step1["num_total_features"]
        p2 = step2["num_total_features"]
        df_diff = p2 - p1
        df_residual = n - p2

        # calculate F-statistic
        f_stat = ((rss_1 - rss_2) / df_diff) / (rss_2 / df_residual)
        p_value = stats.f.sf(f_stat, df_diff, df_residual)

        ftest_results.append({
            "feature_progression": step1["feature_progression"],
            "step1": step1["progression_step"],
            "step2": step2["progression_step"],
            "rss_1": rss_1,
            "rss_2": rss_2,
            "df_diff": df_diff,
            "df_residual": df_residual,
            "f_stat": f_stat,
            "p_value": p_value,
        })

    return ftest_results


def eval_stepwise(feature_block):
    fit_electrodes = sorted(eoi_df[eoi_df.feature_block == feature_block].electrode.values)
    if len(fit_electrodes) == 0:
        return {"cv_results": None, "ftest_results": None}
    cv_results = fit_stepwise(feature_block, fit_electrodes=fit_electrodes)
    ftest_results = ftest_stepwise(feature_block, cv_results)
    
    return {
        "cv_results": cv_results,
        "ftest_results": ftest_results,
    }

In [ ]:
with open(Path(outdir) / "results.pkl", "wb") as f:
    pickle.dump({
        feature_block: eval_stepwise(feature_block)
        for feature_block in feature_progressions
    }, f)